# Regime-Conditional Credit Factor Rotation: A Bayesian Framework for IG Rating-Tier Allocation

---

**Abstract**

This paper tests the hypothesis that **systematic rotation across investment-grade rating tiers (AAA/AA/A/BBB) can generate excess returns over a market-cap-weighted benchmark when conditioned on credit market regimes**. We propose a three-layer Bayesian pipeline — Hidden Markov Model regime detection, Black-Litterman return estimation with Prophet-generated views, and factor-tilted portfolio construction — applied exclusively to publicly available FRED OAS data. We find [results presented in Section 8]. The framework is fully implemented in Python with an interactive Streamlit dashboard.

---

## Table of Contents

1. [Hypothesis & Motivation](#1-hypothesis)
2. [Data Constraints & Design Choices](#2-constraints)
3. [System Architecture](#3-architecture)
4. [Data Foundation: FRED OAS Series](#4-data)
5. [HMM Regime Detection](#5-hmm)
6. [Black-Litterman Return Estimation](#6-bl)
7. [Prophet OAS Forecasting](#7-prophet)
8. [Credit Factor Signals](#8-factors)
9. [Historical Backtest Results](#9-backtest)
10. [Stress Testing Framework](#10-stress)
11. [Monte Carlo Risk Analysis](#11-mc)
12. [Conclusions & Next Steps](#12-conclusions)

In [1]:
# ── Setup ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

# Project imports
from credit_portfolio.config import load_config, resolve_csv_path
from credit_portfolio.data.loader import load
from credit_portfolio.data.constants import (
    DURATIONS, IG_ASSETS, ASSET_LABELS, IG_MARKET_WEIGHTS_ARRAY,
    MARKET_WEIGHTS, COV_WINDOW, BL_RIDGE_PENALTY,
    OAS_PCT_TO_BP, MONTHS_PER_YEAR,
    LAMBDA_RISK_AVERSION, RATINGS,
)

# ── Color Palette ────────────────────────────────────────────────
NAVY      = "#00205B"   # primary — headers, key series
DENIM     = "#0078D4"   # secondary — supporting series
SKY       = "#62B5E5"   # tertiary — fills, light accents
CHARCOAL  = "#4A4A4A"   # text, axes, benchmark lines
SILVER    = "#B0B0B0"   # gridlines, muted elements
GOLD      = "#C4972A"   # accent — warnings, amber signals
CRIMSON   = "#B71C1C"   # negative — stress, losses, risk
SAGE      = "#2E7D32"   # positive — tightening, gains

# Plotly template defaults
TEMPLATE = "plotly_white"

# Load real FRED data
cfg = load_config()
csv_path = resolve_csv_path(cfg)
monthly = load(str(csv_path))

print(f"FRED data loaded: {len(monthly)} monthly observations")
print(f"Date range: {monthly.index[0].strftime('%Y-%m')} to {monthly.index[-1].strftime('%Y-%m')}")
print(f"Columns: {list(monthly.columns)}")

FRED data loaded: 351 monthly observations
Date range: 1997-01 to 2026-03
Columns: ['oas_ig', 'oas_aaa', 'oas_aa', 'oas_a', 'oas_bbb', 'oas_hy', 'oas_bb', 'oas_b', 'oas_ccc', 'oas_1_3y', 'oas_3_5y', 'oas_5_7y', 'oas_7_10y', 'oas_10_15y', 'oas_15py', 'tr_ig', 'tr_bbb', 'tr_hy']


In [2]:
# ── Full Pipeline Data Flow — What Goes Where ────────────────────
flow_html = """
<div style="font-family: 'Helvetica Neue', Arial, sans-serif; max-width: 960px; margin: 30px auto;">

<h2 style="color: #00205B; text-align: center; font-weight: 700; font-size: 22px; margin-bottom: 2px;">
How the Model Works — End-to-End Data Flow
</h2>
<p style="color: #4A4A4A; text-align: center; font-size: 12px; margin-bottom: 30px;">
Every arrow is a real variable. Every box is a real computation. Nothing is assumed — everything is derived from FRED data.
</p>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 1: RAW DATA                                               -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="position: relative;">

  <!-- Step number -->
  <div style="position: absolute; left: 0; top: 0; background: #00205B; color: white;
              width: 28px; height: 28px; border-radius: 50%; text-align: center;
              line-height: 28px; font-weight: 700; font-size: 13px;">1</div>

  <div style="margin-left: 42px; background: #00205B; color: white; padding: 16px 24px;
              border-radius: 4px; margin-bottom: 4px;">
    <div style="font-size: 14px; font-weight: 700;">INPUT: FRED OAS Time Series</div>
    <div style="font-size: 11px; margin-top: 6px; opacity: 0.85; line-height: 1.7;">
      We download <b>monthly Option-Adjusted Spreads</b> from FRED for 4 rating buckets:<br>
      <span style="background: rgba(255,255,255,0.12); padding: 1px 6px; border-radius: 2px;">
        oas_aaa</span> &nbsp;
      <span style="background: rgba(255,255,255,0.12); padding: 1px 6px; border-radius: 2px;">
        oas_aa</span> &nbsp;
      <span style="background: rgba(255,255,255,0.12); padding: 1px 6px; border-radius: 2px;">
        oas_a</span> &nbsp;
      <span style="background: rgba(255,255,255,0.12); padding: 1px 6px; border-radius: 2px;">
        oas_bbb</span>
      <br>
      These are <b>real ICE BofA index spreads</b> in percentage points (e.g., 1.45 = 145bp).
      <br>25+ years of history. This is the <b>only input</b> to the entire system.
    </div>
  </div>

  <!-- Arrow with label -->
  <div style="margin-left: 42px; display: flex; align-items: center; padding: 2px 0 2px 30px;">
    <div style="color: #B0B0B0; font-size: 18px; margin-right: 8px;">&#9660;</div>
    <div style="color: #4A4A4A; font-size: 10px; font-style: italic;">
      monthly DataFrame with DatetimeIndex &rarr; feeds into 3 parallel processes
    </div>
  </div>
</div>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 2: THREE PARALLEL PROCESSES                               -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="display: flex; gap: 8px; margin-bottom: 4px;">

  <!-- 2A: HMM -->
  <div style="flex: 1; position: relative;">
    <div style="position: absolute; left: 0; top: 0; background: #00205B; color: white;
                width: 24px; height: 24px; border-radius: 50%; text-align: center;
                line-height: 24px; font-weight: 700; font-size: 11px;">2a</div>
    <div style="margin-left: 0; margin-top: 0; background: #F5F7FA; border-left: 4px solid #00205B;
                padding: 34px 12px 12px; border-radius: 2px; min-height: 280px;">
      <div style="color: #00205B; font-weight: 700; font-size: 13px; margin-bottom: 8px;">
        HMM Regime Detection
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #00205B; font-size: 10px; font-weight: 700; margin-bottom: 3px;">TAKES IN:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          &bull; OAS<sub>IG</sub> composite level<br>
          &bull; &Delta;OAS 1-month change<br>
          &bull; &Delta;OAS 3-month change<br>
          &bull; &Delta;OAS 6-month change
        </div>
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #00205B; font-size: 10px; font-weight: 700; margin-bottom: 3px;">DOES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          Fits a <b>3-state Gaussian HMM</b> via EM algorithm (300 iter).
          Each state has its own mean vector &amp; covariance.
          The Viterbi path assigns every month to a regime.
        </div>
      </div>

      <div style="background: #E8F0FE; border: 1px solid #C4D7F2; border-radius: 2px;
                  padding: 8px 10px;">
        <div style="color: #00205B; font-size: 10px; font-weight: 700; margin-bottom: 3px;">PRODUCES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          <b>regime</b> = Compression / Normal / Stress<br>
          <b>&tau;</b> = 0.010 / 0.025 / 0.075<br>
          <b>&omega;</b> = 0.5 / 1.0 / 3.0<br>
          <span style="font-size: 9px; color: #888;">&tau; controls prior confidence, &omega; controls view uncertainty</span>
        </div>
      </div>
    </div>
  </div>

  <!-- 2B: Prophet -->
  <div style="flex: 1; position: relative;">
    <div style="position: absolute; left: 0; top: 0; background: #0078D4; color: white;
                width: 24px; height: 24px; border-radius: 50%; text-align: center;
                line-height: 24px; font-weight: 700; font-size: 11px;">2b</div>
    <div style="margin-left: 0; margin-top: 0; background: #F5F7FA; border-left: 4px solid #0078D4;
                padding: 34px 12px 12px; border-radius: 2px; min-height: 280px;">
      <div style="color: #00205B; font-weight: 700; font-size: 13px; margin-bottom: 8px;">
        Prophet OAS Forecasting
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #0078D4; font-size: 10px; font-weight: 700; margin-bottom: 3px;">TAKES IN:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          &bull; Full OAS history for <b>each bucket</b><br>
          &bull; oas_aaa, oas_aa, oas_a, oas_bbb<br>
          &bull; Each fitted separately
        </div>
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #0078D4; font-size: 10px; font-weight: 700; margin-bottom: 3px;">DOES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          Fits <b>logistic-growth Prophet</b> model:<br>
          OAS(t) = g(t) + s(t) + &epsilon;(t)<br>
          Cap = historical max &times; 2.5<br>
          Forecasts 3 months ahead with uncertainty
        </div>
      </div>

      <div style="background: #E8F0FE; border: 1px solid #C4D7F2; border-radius: 2px;
                  padding: 8px 10px;">
        <div style="color: #0078D4; font-size: 10px; font-weight: 700; margin-bottom: 3px;">PRODUCES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          Per bucket:<br>
          <b>&Delta;OAS</b> = forecast &minus; current (in %)<br>
          <b>expected_return</b> = carry + price return<br>
          <span style="font-size: 9px; color: #888;">
            r = OAS/10000 &times; h/12 + D &times; (&minus;&Delta;OAS/10000) &times; h/12
          </span>
        </div>
      </div>
    </div>
  </div>

  <!-- 2C: Covariance -->
  <div style="flex: 1; position: relative;">
    <div style="position: absolute; left: 0; top: 0; background: #62B5E5; color: white;
                width: 24px; height: 24px; border-radius: 50%; text-align: center;
                line-height: 24px; font-weight: 700; font-size: 11px;">2c</div>
    <div style="margin-left: 0; margin-top: 0; background: #F5F7FA; border-left: 4px solid #62B5E5;
                padding: 34px 12px 12px; border-radius: 2px; min-height: 280px;">
      <div style="color: #00205B; font-weight: 700; font-size: 13px; margin-bottom: 8px;">
        Covariance &amp; Equilibrium
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #62B5E5; font-size: 10px; font-weight: 700; margin-bottom: 3px;">TAKES IN:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          &bull; Monthly spread returns per bucket<br>
          &bull; r &asymp; &minus;D &times; &Delta;OAS + OAS/12<br>
          &bull; Market-cap weights w<sub>mkt</sub><br>
          <span style="font-size: 9px; color: #888;">AAA 4% / AA 12% / A 34% / BBB 50%</span>
        </div>
      </div>

      <div style="background: white; border: 1px solid #E8E8E8; border-radius: 2px;
                  padding: 8px 10px; margin-bottom: 8px;">
        <div style="color: #62B5E5; font-size: 10px; font-weight: 700; margin-bottom: 3px;">DOES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          <b>&Sigma;</b> = 60-month rolling covariance<br>
          <b>&pi;</b> = &lambda; &times; &Sigma; &times; w<sub>mkt</sub><br>
          <span style="font-size: 9px; color: #888;">&lambda; = 2.5 (risk aversion)</span><br>
          This is what the market <i>implies</i> returns should be
        </div>
      </div>

      <div style="background: #E8F0FE; border: 1px solid #C4D7F2; border-radius: 2px;
                  padding: 8px 10px;">
        <div style="color: #62B5E5; font-size: 10px; font-weight: 700; margin-bottom: 3px;">PRODUCES:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.6;">
          <b>&Sigma;</b> = 4&times;4 covariance matrix<br>
          <b>&pi;</b> = equilibrium returns (prior)<br>
          <span style="font-size: 9px; color: #888;">These are the "anchor" — what the market believes</span>
        </div>
      </div>
    </div>
  </div>
</div>

<!-- Arrows converging -->
<div style="text-align: center; padding: 2px 0; color: #B0B0B0; font-size: 10px;">
  <span style="color: #00205B;">&#9660; &tau;, &omega;</span>
  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
  <span style="color: #0078D4;">&#9660; Q (views)</span>
  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
  <span style="color: #62B5E5;">&#9660; &pi;, &Sigma;</span>
  &nbsp;&nbsp;&nbsp;&nbsp;all three feed into &darr;
</div>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 3: BLACK-LITTERMAN                                        -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="position: relative; margin-bottom: 4px;">
  <div style="position: absolute; left: 0; top: 0; background: #00205B; color: white;
              width: 28px; height: 28px; border-radius: 50%; text-align: center;
              line-height: 28px; font-weight: 700; font-size: 13px;">3</div>

  <div style="margin-left: 42px; background: linear-gradient(135deg, #00205B, #0078D4);
              color: white; padding: 18px 24px; border-radius: 4px;">
    <div style="font-size: 14px; font-weight: 700; margin-bottom: 8px;">
      BLACK-LITTERMAN: Combining Prior + Views + Regime
    </div>
    <div style="display: flex; gap: 16px;">
      <div style="flex: 1;">
        <div style="font-size: 10px; font-weight: 700; opacity: 0.8; margin-bottom: 4px;">INPUTS ARRIVING:</div>
        <div style="font-size: 11px; line-height: 1.7;">
          &bull; <b>&pi;</b> (prior returns) from Step 2c<br>
          &bull; <b>Q</b> (Prophet forecast views) from Step 2b<br>
          &bull; <b>&Sigma;</b> (covariance) from Step 2c<br>
          &bull; <b>&tau;</b> (prior uncertainty) from HMM Step 2a<br>
          &bull; <b>&omega;</b> (view uncertainty scale) from HMM Step 2a
        </div>
      </div>
      <div style="flex: 1;">
        <div style="font-size: 10px; font-weight: 700; opacity: 0.8; margin-bottom: 4px;">THE MATH:</div>
        <div style="font-size: 11px; line-height: 1.7;">
          &Omega; = diag(P &times; &tau;&Sigma; &times; P') &times; &omega;<br>
          &mu;<sub>BL</sub> = [(&tau;&Sigma;)<sup>&minus;1</sup> + P'&Omega;<sup>&minus;1</sup>P]<sup>&minus;1</sup>
          &times; [(&tau;&Sigma;)<sup>&minus;1</sup>&pi; + P'&Omega;<sup>&minus;1</sup>Q]<br>
          <span style="opacity: 0.7; font-size: 10px;">
            In <b>Stress</b>: high &tau; + high &omega; &rarr; posterior stays near &pi; (defensive)<br>
            In <b>Compression</b>: low &tau; + low &omega; &rarr; posterior incorporates Q (views matter)
          </span>
        </div>
      </div>
      <div style="flex: 0.6;">
        <div style="font-size: 10px; font-weight: 700; opacity: 0.8; margin-bottom: 4px;">PRODUCES:</div>
        <div style="font-size: 11px; line-height: 1.7;">
          <b>&mu;<sub>BL</sub></b> = posterior expected returns<br>
          One number per bucket:<br>
          &mu;<sub>AAA</sub>, &mu;<sub>AA</sub>, &mu;<sub>A</sub>, &mu;<sub>BBB</sub>
        </div>
      </div>
    </div>
  </div>
</div>

<!-- Arrow -->
<div style="margin-left: 42px; display: flex; align-items: center; padding: 2px 0 2px 30px;">
  <div style="color: #B0B0B0; font-size: 18px; margin-right: 8px;">&#9660;</div>
  <div style="color: #4A4A4A; font-size: 10px; font-style: italic;">
    &mu;<sub>BL</sub> posterior returns &rarr; inform which buckets to overweight
  </div>
</div>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 4: PORTFOLIO CONSTRUCTION                                 -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="position: relative; margin-bottom: 4px;">
  <div style="position: absolute; left: 0; top: 0; background: #00205B; color: white;
              width: 28px; height: 28px; border-radius: 50%; text-align: center;
              line-height: 28px; font-weight: 700; font-size: 13px;">4</div>

  <div style="margin-left: 42px; background: #F5F7FA; border: 2px solid #00205B;
              padding: 16px 24px; border-radius: 4px;">
    <div style="color: #00205B; font-size: 14px; font-weight: 700; margin-bottom: 8px;">
      PORTFOLIO CONSTRUCTION: Tilt Weights by Signal
    </div>
    <div style="display: flex; gap: 16px;">
      <div style="flex: 1;">
        <div style="color: #00205B; font-size: 10px; font-weight: 700; margin-bottom: 4px;">
          FACTOR SIGNALS (computed from same FRED data):</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.7;">
          <b>DTS</b> (50%): Duration &times; Spread — cross-sectional z-score<br>
          <b>Value</b> (25%): log(OAS) — cross-sectional z-score<br>
          <b>Momentum</b> (25%): trailing 6-month cumulative excess return<br>
          z<sub>composite</sub> = 0.50 &times; z<sub>DTS</sub> + 0.25 &times; z<sub>Value</sub> + 0.25 &times; z<sub>Mom</sub>
        </div>
      </div>
      <div style="flex: 1;">
        <div style="color: #00205B; font-size: 10px; font-weight: 700; margin-bottom: 4px;">
          WEIGHT CONSTRUCTION:</div>
        <div style="color: #4A4A4A; font-size: 10px; line-height: 1.7;">
          w<sub>i</sub> = w<sub>mkt,i</sub> + &alpha; &times; z<sub>composite,i</sub> / &Sigma;|z|<br>
          <span style="color: #888; font-size: 9px;">
            &alpha; = 10% tilt strength &nbsp;|&nbsp; Floor each weight at 1%<br>
            Renormalize to sum to 100% &nbsp;|&nbsp; TC drag 5bp one-way
          </span><br><br>
          <b>Benchmark:</b> AAA 4% / AA 12% / A 34% / BBB 50%<br>
          Strategy tilts &plusmn; around these based on signal strength
        </div>
      </div>
    </div>
  </div>
</div>

<!-- Arrow -->
<div style="margin-left: 42px; display: flex; align-items: center; padding: 2px 0 2px 30px;">
  <div style="color: #B0B0B0; font-size: 18px; margin-right: 8px;">&#9660;</div>
  <div style="color: #4A4A4A; font-size: 10px; font-style: italic;">
    final weights w<sub>AAA</sub>, w<sub>AA</sub>, w<sub>A</sub>, w<sub>BBB</sub> &rarr; execute monthly
  </div>
</div>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 5: OUTPUT                                                 -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="position: relative; margin-bottom: 4px;">
  <div style="position: absolute; left: 0; top: 0; background: #00205B; color: white;
              width: 28px; height: 28px; border-radius: 50%; text-align: center;
              line-height: 28px; font-weight: 700; font-size: 13px;">5</div>

  <div style="margin-left: 42px; background: #00205B; color: white;
              padding: 14px 24px; border-radius: 4px;">
    <div style="font-size: 14px; font-weight: 700; margin-bottom: 4px;">
      OUTPUT: Monthly Portfolio Allocation
    </div>
    <div style="font-size: 11px; opacity: 0.85; line-height: 1.6;">
      Four weights summing to 100% — how much to allocate to each IG rating bucket.
      Rebalanced monthly. Compared against market-cap benchmark.
    </div>
  </div>
</div>

<!-- Arrow -->
<div style="margin-left: 42px; display: flex; align-items: center; padding: 2px 0 2px 30px;">
  <div style="color: #B0B0B0; font-size: 18px; margin-right: 8px;">&#9660;</div>
  <div style="color: #4A4A4A; font-size: 10px; font-style: italic;">
    allocation is then stress-tested, simulated, and backtested
  </div>
</div>

<!-- ═══════════════════════════════════════════════════════════════ -->
<!-- STEP 6: VALIDATION                                             -->
<!-- ═══════════════════════════════════════════════════════════════ -->
<div style="position: relative;">
  <div style="position: absolute; left: 0; top: 0; background: #C4972A; color: white;
              width: 28px; height: 28px; border-radius: 50%; text-align: center;
              line-height: 28px; font-weight: 700; font-size: 13px;">6</div>

  <div style="margin-left: 42px; display: flex; gap: 8px;">
    <div style="flex: 1; background: #FAFAFA; border: 1px solid #E0E0E0; padding: 12px;
                border-radius: 2px;">
      <div style="color: #C4972A; font-weight: 700; font-size: 11px; margin-bottom: 4px;">
        HISTORICAL BACKTEST</div>
      <div style="color: #4A4A4A; font-size: 10px; line-height: 1.5;">
        Run the full pipeline on 25yr FRED history.<br>
        Measure: Sharpe, alpha, IR, max drawdown.<br>
        <b>No look-ahead</b> — signals use only past data.
      </div>
    </div>
    <div style="flex: 1; background: #FAFAFA; border: 1px solid #E0E0E0; padding: 12px;
                border-radius: 2px;">
      <div style="color: #C4972A; font-weight: 700; font-size: 11px; margin-bottom: 4px;">
        STRESS TESTING</div>
      <div style="color: #4A4A4A; font-size: 10px; line-height: 1.5;">
        Shock OAS by +100 to +300bp.<br>
        Recompute signals &amp; weights under stress.<br>
        Measure: price impact, weight shifts.
      </div>
    </div>
    <div style="flex: 1; background: #FAFAFA; border: 1px solid #E0E0E0; padding: 12px;
                border-radius: 2px;">
      <div style="color: #C4972A; font-weight: 700; font-size: 11px; margin-bottom: 4px;">
        MONTE CARLO</div>
      <div style="color: #4A4A4A; font-size: 10px; line-height: 1.5;">
        Resample from real backtest returns.<br>
        5,000 simulations &times; 24-month paths.<br>
        Measure: VaR, CVaR, P(Loss).
      </div>
    </div>
  </div>
</div>

<!-- Footer -->
<div style="text-align: center; margin-top: 20px; padding-top: 12px; border-top: 1px solid #E0E0E0;">
  <span style="color: #B0B0B0; font-size: 10px;">
    All data sourced from FRED (Federal Reserve Economic Data) &nbsp;&bull;&nbsp;
    ICE BofA OAS indices &nbsp;&bull;&nbsp;
    No synthetic bonds or simulated inputs anywhere in the pipeline
  </span>
</div>

</div>
"""

display(HTML(flow_html))

---

## 1. Hypothesis & Motivation <a id='1-hypothesis'></a>

### The Central Hypothesis

> **H₁: Systematic rotation across IG rating tiers, conditioned on a regime-switching model, generates statistically significant excess returns over a market-cap-weighted IG benchmark after transaction costs.**

Stated plainly: we believe that by identifying *which credit regime we are in* (tight spreads vs. normal vs. stress), we can **tilt portfolio weights toward the rating buckets that are most likely to outperform over the next month**, and that doing this systematically beats a passive allocation.

### What We Are NOT Claiming

This is important for intellectual honesty:

- We are **not** claiming to pick individual bonds. We operate at the rating-tier level only (AAA, AA, A, BBB).
- We are **not** claiming to time the market. The regime model identifies *current conditions*, not future turning points.
- We are **not** claiming novel factor discovery. DTS, value, and momentum are well-documented in the credit literature (Ben Dor et al., 2012; Israel et al., 2018; Houweling & van Zundert, 2017).
- We **are** claiming that combining these factors with regime conditioning and Bayesian return estimation improves upon naive factor rotation.

### The Economic Intuition

The investment-grade credit market has a structural feature that creates opportunity:

**BBB bonds carry ~100-130bp more spread than AAA bonds.** This "credit risk premium" compensates for higher default and downgrade risk. Most of the time, harvesting this premium is rational — IG default rates are historically <0.5% annually. But during stress episodes (GFC, COVID), BBB spreads can blow out by 300-500bp while AAA spreads barely move. A portfolio overweight BBB at the wrong time suffers catastrophic drawdowns.

**The key question is therefore: can we systematically identify when to harvest the BBB premium and when to rotate toward quality?**

Our hypothesis is that the answer is yes, via three mechanisms:

| Mechanism | How It Helps | When It Matters Most |
|-----------|-------------|---------------------|
| **Regime detection** (HMM) | Identifies compression/normal/stress before it's obvious | Transitions — the 10-15% of months where regimes shift |
| **Spread forecasting** (Prophet) | Predicts directional OAS moves per bucket | Normal markets — where mean-reversion signals are strongest |
| **Bayesian blending** (BL) | Regime-adjusts how much we trust forecasts vs equilibrium | Stress — when forecast reliability degrades |

### The Risk-Return Tradeoff We Are Exploiting

The specific tradeoff is:

- **The return source**: the credit risk premium (BBB-AAA spread) and mean-reversion in relative spreads
- **The risk**: getting caught overweight BBB during a stress event, or whipsawing during regime transitions
- **Our edge**: regime conditioning reduces the probability of being overweight BBB at the worst time, while Bayesian blending prevents the model from over-reacting to noisy signals

### Why This Matters

Most active IG credit managers rely on discretionary judgment to make these rotation decisions. We ask: can a systematic, transparent, fully auditable framework do comparably well? Even if the alpha is modest (this is IG credit, not equities), the value lies in:

1. **Discipline** — rules-based allocation removes behavioral biases
2. **Transparency** — every weight traces to observable factor scores and regime states
3. **Scalability** — runs on free, publicly available FRED data
4. **Risk management** — stress tests and Monte Carlo are built into the framework, not afterthoughts

---

## 2. Data Constraints & Design Choices <a id='2-constraints'></a>

### The Data We Have — and Don't Have

This section is critical. Every design choice in this system traces back to a data constraint. We are transparent about what we lack and how it shaped the architecture.

**What we have:**
- Monthly OAS (Option-Adjusted Spread) for 4 IG rating tiers from FRED
- ICE BofA index-level data: AAA, AA, A, BBB
- 25+ years of history (1997–present)
- Free, publicly available, institutional-quality

**What we do NOT have:**
- Bond-level data (TRACE, Bloomberg)
- Issuer-level spreads, coupons, maturities
- Sector breakdowns within rating tiers
- Intraday or daily OAS (our data is monthly)
- Total return indices at the rating tier level (we approximate)

### How Data Constraints Drove Architecture Decisions

| Constraint | Consequence | Our Solution |
|-----------|------------|-------------|
| **Only 4 assets** (rating buckets) | Cannot run mean-variance optimization — 4 assets with a 4×4 covariance matrix is underdetermined | Black-Litterman, which starts from market equilibrium and adjusts — doesn't need optimization |
| **No bond-level data** | Cannot compute issuer-level credit signals (CDS basis, relative value) | Use bucket-level proxies: DTS, value (log OAS), momentum on bucket returns |
| **Monthly frequency** | Cannot capture intra-month spread moves or trade timing | Monthly rebalancing with transaction cost drag — realistic for institutional IG portfolios |
| **No total return index** | Cannot observe actual portfolio returns directly | Approximate: return ≈ carry (OAS/12) + price return (−D × ΔOAS) — standard in credit literature |
| **Spreads are non-stationary** | OAS levels trend and regime-shift — simple time-series models fail | HMM to identify regimes; Prophet with logistic growth cap for non-stationary forecasting |
| **Small cross-section** | With only 4 buckets, cross-sectional z-scores are noisy (n=4) | Combine multiple signals (DTS, value, momentum) and use modest tilt strength (α=10%) to avoid overfitting |

### Why Black-Litterman Instead of [Alternative]?

**Why not mean-variance optimization?**
With 4 assets, MVO produces extreme corner solutions (100% in one bucket). BL solves this by starting from market-implied returns and making incremental adjustments.

**Why not a pure factor model?**
Factor models require a large cross-section to be statistically meaningful. With n=4 rating buckets, a pure z-score-based strategy is too noisy. BL provides a Bayesian anchor (the equilibrium) that regularizes the signal.

**Why not a simple momentum/carry strategy?**
Because regime matters. A carry strategy that overweights BBB in 2007 or February 2020 gets destroyed. The HMM layer adds regime conditioning that a simple strategy lacks.

**Why not machine learning (XGBoost, neural nets)?**
With 4 assets and ~300 monthly observations, there is not enough data to train any ML model without severe overfitting. The BL framework is parametric with ~10 free parameters — appropriate for the data budget.

### Why Prophet for Forecasting?

Prophet is chosen because:
1. **Handles non-stationarity** — logistic growth mode with a cap prevents forecasting negative OAS or infinite spreads
2. **Additive decomposition** — separates trend from seasonality, which is economically interpretable
3. **Uncertainty quantification** — provides forecast intervals, not just point estimates
4. **Robust to missing data** — handles the occasional FRED gap gracefully
5. **Fast** — fits in <10s per bucket vs. hours for MCMC-based alternatives

### Limitations We Acknowledge Upfront

1. **Cross-sectional factors (DTS, Value) are partly mechanical** — BBB almost always has higher DTS and wider spreads than AAA. The time-varying component is what generates signal, not the level.
2. **Return approximation** — our carry + duration × ΔOAS formula ignores convexity, roll-down, and coupon effects. This is standard in the academic literature but introduces tracking error vs. actual index returns.
3. **No credit events** — we do not model defaults, downgrades, or fallen angels. IG default rates are low enough (~0.1% annually) that this is acceptable for a rating-tier rotation strategy.
4. **Look-ahead in HMM training** — the HMM is fit on the full history, which introduces mild look-ahead bias in regime labels. A production system would use expanding-window refitting.

These limitations are discussed further in Section 12.

---

## 2.5 Prior Work — What Others Have Done <a id='2.5-literature'></a>

### Credit Factor Investing

The idea that systematic factors explain credit returns is well-established:

- **Ben Dor, Dynkin, Hyman, Houweling, van Leeuwen, Penninga (2012)** — *Systematic Credit Investing*: demonstrated that carry, value, and momentum factors explain a significant portion of cross-sectional credit returns. Showed that multi-factor models outperform single-factor strategies in IG.
- **Israel, Palhares, Richardson (2018)** — *Common factors in corporate bond returns*: identified market, credit, and liquidity factors analogous to equity Fama-French factors. Found that momentum and value premiums exist in credit but are weaker than in equities.
- **Houweling & van Zundert (2017)** — *Factor investing in the corporate bond market*: demonstrated size, low-risk, value, and momentum factors in corporate bonds. Critically, they showed these factors work at the **bond level** with hundreds of securities — our 4-bucket setting is much more constrained.
- **Bai, Bali, Wen (2019)** — *Common risk factors in the cross-section of corporate bond returns*: DTS (Duration Times Spread) identified as the dominant single factor in explaining credit return dispersion.

### Regime-Switching in Credit Markets

- **Hamilton (1989)** — foundational HMM framework for economic regime detection, later applied to credit spreads.
- **Ang & Bekaert (2002)** — regime-switching models for asset allocation; showed that ignoring regimes leads to suboptimal portfolios.
- **Guidolin & Timmermann (2007)** — demonstrated that regime-switching models improve out-of-sample portfolio allocation in multi-asset settings. The key insight: **parameters that work in normal markets fail in stress**.

### Black-Litterman in Fixed Income

- **Black & Litterman (1992)** — original framework, developed for equities at Goldman Sachs.
- **Meucci (2010)** — extended BL to general distributions and applied to fixed income. Key contribution: regime-dependent view uncertainty (ω scaling).
- **Fabozzi, Focardi, Jonas (2010)** — applied BL to bond portfolio construction, demonstrating its advantages over MVO for small asset universes.

### Where We Differ From Prior Work

| Prior Work | Their Setting | Our Setting | Key Difference |
|-----------|--------------|-------------|----------------|
| Ben Dor et al. (2012) | 1000+ bonds, sector + rating | 4 rating buckets | We lack cross-sectional depth — compensate with Bayesian regularization |
| Israel et al. (2018) | Bond-level TRACE data | FRED index-level OAS | We use free public data — replicable by anyone |
| Houweling & van Zundert (2017) | Multi-factor long/short | Long-only rotation | Constrained setting — floors at 1%, tilt ≤10% |
| Guidolin & Timmermann (2007) | Multi-asset (equity/bond/cash) | Within-credit rotation | Same idea, narrower universe |
| Meucci (2010) | General BL framework | BL + HMM + Prophet | We combine three specific models into a pipeline |

**Our contribution is not in any single model, but in the pipeline**: regime detection calibrating BL parameters, which incorporates Prophet-generated views, which feeds factor-tilted allocation — all on a minimal, reproducible dataset.

---

## 2.7 Experiment Design <a id='2.7-experiment'></a>

### What We Test

We conduct a **single historical backtest** over the full FRED sample (1997–present) comparing:

- **Strategy**: monthly rebalanced factor-tilted rotation (DTS 50%, Value 25%, Momentum 25%) with α=10% tilt strength
- **Benchmark**: static market-cap-weighted allocation (AAA 4%, AA 12%, A 34%, BBB 50%)

### The Rules

These rules are fixed *before* the backtest. There is no optimization of rules after seeing results.

1. **Rebalancing**: monthly, on the last business day
2. **Signal computation**: uses only data available at time *t* (no look-ahead)
3. **Weight construction**: w_i = w_mkt + α × z_composite / Σ|z|, floored at 1%, renormalized
4. **Transaction costs**: 5bp one-way on turnover (conservative for institutional IG)
5. **Factor weights**: DTS 50%, Value 25%, Momentum 25% (from credit literature, not optimized)
6. **Momentum window**: 6 months trailing (standard in credit momentum literature)
7. **Tilt strength**: α = 10% (chosen to be meaningful but not extreme for a 4-asset rotation)

### What We Measure

| Metric | What It Tells Us | Significance Threshold |
|--------|-----------------|----------------------|
| **Annualized alpha** | Excess return over benchmark | >0 after costs |
| **Information ratio** | Alpha per unit of tracking error | >0.3 considered meaningful |
| **t-stat of alpha** | Statistical significance of alpha | >1.96 (95% confidence) |
| **Hit rate** | % of months strategy outperforms | >50% |
| **Sharpe ratio** | Risk-adjusted absolute return | >benchmark Sharpe |
| **Max drawdown** | Worst peak-to-trough loss | <benchmark drawdown |

### How We Interpret Results

- If alpha > 0 with t-stat > 1.96: **hypothesis supported** — regime-conditional rotation adds value
- If alpha > 0 but t-stat < 1.96: **inconclusive** — positive but not statistically significant
- If alpha ≤ 0: **hypothesis rejected** — the factor signals do not add value after costs
- We also examine **conditional performance**: does the strategy outperform in regime transitions (where the HMM should add most value)?

### What Follows the Backtest

After the historical backtest, we validate robustness via:
1. **Stress testing** — apply extreme OAS shocks and check that the model doesn't blow up
2. **Monte Carlo simulation** — bootstrap from real returns to estimate forward-looking risk (VaR, CVaR)
3. These are NOT separate tests of the hypothesis — they are robustness checks on the same strategy

---

## 3. System Architecture <a id='3-architecture'></a>

### Three-Layer Bayesian Pipeline

The system operates as a **pipeline of three Bayesian layers**, each adding information. The interactive flow chart above shows the full data flow — from raw FRED OAS inputs through regime detection, return estimation, and portfolio construction to final allocation weights.

### Key Design Principles

1. **Real data only** — all inputs are ICE BofA indices published on FRED
2. **No look-ahead bias** — signals at time *t* use only data available at *t*
3. **Regime-adaptive** — BL parameters (τ, ω) shift with HMM state
4. **Transparent** — every allocation decision traces to observable factor scores

### Information Flow

| Layer | Input | Model | Output | Feeds Into |
|-------|-------|-------|--------|------------|
| 1 — Regime | OAS level + 1/3/6m changes | 3-state Gaussian HMM | Regime label, τ, ω | Layer 2 (BL calibration) |
| 2 — Returns | FRED OAS, Prophet forecasts | Black-Litterman | μ_BL posterior returns | Layer 3 (expected returns) |
| 3 — Allocation | Factor signals, μ_BL | Z-score tilt + constraints | w_AAA, w_AA, w_A, w_BBB | Trading desk |

### Update Frequency

- **HMM regime**: recalculated at each monthly rebalance — uses expanding window of all available history
- **Prophet forecasts**: refit at each rebalance — logistic-growth model with yearly seasonality (~30s per bucket)
- **BL posterior**: computed fresh each month — incorporates latest regime (τ, ω) and Prophet views (Q)
- **Factor signals**: computed monthly — DTS and Value use current OAS snapshot, Momentum uses trailing 6-month returns
- **Portfolio weights**: rebalanced monthly with 5bp one-way transaction cost drag

---

## 4. Data Foundation: FRED OAS Series <a id='4-data'></a>

All data is sourced from the **Federal Reserve Economic Data (FRED)** database, published by ICE BofA. These are the official benchmark indices used by institutional credit investors globally.

In [3]:
# ── Figure 1: Historical OAS by Rating Bucket ────────────────────
fig = go.Figure()

oas_cols = {'oas_aaa': ('AAA', NAVY), 'oas_aa': ('AA', DENIM),
            'oas_a': ('A', SKY), 'oas_bbb': ('BBB', GOLD)}

for col, (label, color) in oas_cols.items():
    if col in monthly.columns:
        fig.add_trace(go.Scatter(
            x=monthly.index, y=monthly[col],
            name=label, line=dict(color=color, width=1.5),
        ))

# Annotate key events
events = [
    ('2001-09-01', '9/11'), ('2008-09-15', 'Lehman'),
    ('2020-03-01', 'COVID'), ('2022-03-01', 'Fed Hike'),
]
for date, label in events:
    fig.add_vline(x=date, line_dash='dot', line_color=SILVER, line_width=0.5)
    fig.add_annotation(x=date, y=1.0, yref='paper', text=label,
                       showarrow=False, font=dict(size=9, color=CHARCOAL))

fig.update_layout(
    title='Figure 1: ICE BofA IG OAS by Rating Tier (FRED)',
    yaxis_title='Option-Adjusted Spread (%)',
    template=TEMPLATE, height=500,
    legend=dict(orientation='h', y=-0.12),
    hovermode='x unified',
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig.show()

In [4]:
# ── Summary statistics table ─────────────────────────────────────
summary_rows = []
for col, (label, _) in oas_cols.items():
    if col in monthly.columns:
        s = monthly[col].dropna()
        summary_rows.append({
            'Rating': label,
            'Observations': len(s),
            'Current OAS (%)': f'{s.iloc[-1]:.2f}',
            'Median OAS (%)': f'{s.median():.2f}',
            'Min OAS (%)': f'{s.min():.2f}',
            'Max OAS (%)': f'{s.max():.2f}',
            'Std Dev (%)': f'{s.std():.2f}',
            'Duration (yr)': DURATIONS.get(col, 'N/A'),
            'Market Weight': f"{MARKET_WEIGHTS.get(label, 0):.0%}",
        })

display(Markdown('### Table 1: FRED OAS Summary Statistics'))
display(pd.DataFrame(summary_rows).set_index('Rating'))

### Table 1: FRED OAS Summary Statistics

,Observations,Current OAS (%),Median OAS (%),Min OAS (%),Max OAS (%),Std Dev (%),Duration (yr),Market Weight
Rating,,,,,,,,
AAA,351,0.45,0.64,0.25,4.07,0.47,8.5,4%
AA,351,0.60,0.77,0.38,4.96,0.62,7.5,12%
A,351,0.78,1.04,0.50,6.37,0.80,7.2,34%
BBB,351,1.15,1.67,0.74,7.84,1.00,6.5,50%


### Key Observations

- **BBB dominates** the IG index at ~50% market weight — this structural feature means BBB spread moves drive index-level performance
- **Spread dispersion** increases dramatically during stress (GFC: BBB peaked >8%, AAA stayed <2%)
- **Mean-reversion** is evident: extreme spreads tend to compress back toward long-run medians over 6-12 month horizons
- The **BBB-AAA spread differential** widens during stress and compresses during risk-on — this is the primary rotation signal

---

## 5. HMM Regime Detection <a id='5-hmm'></a>

### Why Regimes Matter

> **Credit markets exhibit three distinct behavioral regimes.** Model parameters that work in normal markets fail catastrophically in stress. The HMM identifies which regime we're in *before* it's obvious.

### What the HMM Is

A **Hidden Markov Model** is a probabilistic model that assumes the observed data (OAS levels and changes) is generated by an underlying hidden process that switches between a finite number of states. We don't observe the state directly — we infer it from the data.

- **"Hidden"**: the regime (Compression/Normal/Stress) is not directly observable
- **"Markov"**: the probability of transitioning to the next state depends only on the current state, not the full history
- **"Model"**: each state has its own Gaussian distribution (mean + covariance) for the observed features

### How It Works in Our System

**Input features** (computed from FRED OAS):
- OAS level (current IG composite spread)
- 1-month OAS change (short-term momentum)
- 3-month OAS change (medium-term trend)
- 6-month OAS change (long-term trend)

**Training**: Expectation-Maximization algorithm with 300 iterations on the full OAS history. The EM algorithm alternates between (E) computing the probability of each state given the data and current parameters, and (M) updating the parameters to maximize likelihood.

**Output**: For each month in history, the model produces:
1. A **state assignment** (0, 1, or 2)
2. A **probability vector** [p_compression, p_normal, p_stress] — how confident the model is
3. A **transition matrix** — the probability of moving between states

### What the Three States Represent

| State | Label | Characteristics | τ (BL) | ω Scale | What It Means for the Portfolio |
|-------|-------|-----------------|--------|---------|-------------------------------|
| 0 | **Compression** | Tight spreads, low vol, mean-reverting | 0.010 | 0.5 | Trust the equilibrium AND the forecasts. Spreads are stable. Lean into factor signals. |
| 1 | **Normal** | Average spreads, moderate vol | 0.025 | 1.0 | Balanced — prior and views weighted equally. Standard factor rotation. |
| 2 | **Stress** | Wide spreads, high vol, fat tails | 0.075 | 3.0 | Don't trust forecasts (ω=3 inflates view uncertainty). Don't trust equilibrium either (τ=0.075). Go defensive — rotate toward quality. |

### Why τ and ω Change By Regime

**τ (tau)** controls how uncertain we are about the equilibrium prior. Higher τ = more uncertain = posterior can deviate more from equilibrium. In stress, τ is high because the "normal" equilibrium is unreliable.

**ω (omega)** scales the uncertainty of our Prophet views. Higher ω = we trust Prophet forecasts less. In stress, ω=3.0 because Prophet (trained on mostly normal data) is unreliable during dislocations.

In [5]:
# ── Figure 2: HMM Regime Detection ───────────────────────────────
from credit_portfolio.models.hmm_regime import fit_hmm, get_current_regime, regime_summary

hmm_result = fit_hmm(monthly, n_states=3, n_iter=300)
regime_info = get_current_regime(hmm_result)

# Map regime label to its probability
_regime_prob = {
    "COMPRESSION": regime_info.get("p_compression", 0),
    "NORMAL": regime_info.get("p_normal", 0),
    "STRESS": regime_info.get("p_stress", 0),
}.get(regime_info["regime"], 0)

print(f"Current regime: {regime_info['regime']} (probability: {_regime_prob:.1%})")
print(f"Omega scale: {hmm_result.omega_scale}")

# Regime probabilities stacked area
probs = hmm_result.state_probs
fig_hmm = go.Figure()

regime_config = {
    0: ('Compression', DENIM),
    1: ('Normal', SILVER),
    2: ('Stress', CRIMSON),
}

for state in range(3):
    if state in probs.columns:
        name, color = regime_config[state]
        fig_hmm.add_trace(go.Scatter(
            x=probs.index, y=probs[state].values,
            name=name, stackgroup='one',
            line=dict(width=0.5, color=color),
        ))

for date, label in events:
    fig_hmm.add_vline(x=date, line_dash='dot', line_color=CHARCOAL, line_width=0.5)

fig_hmm.update_layout(
    title='Figure 2: HMM Regime Probabilities (FRED IG OAS)',
    yaxis_title='State Probability', yaxis_range=[0, 1],
    template=TEMPLATE, height=450,
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_hmm.show()

Current regime: COMPRESSION (probability: 100.0%)
Omega scale: 0.5


In [6]:
# ── Table 2: Regime Statistics ────────────────────────────────────
display(Markdown('### Table 2: Regime Statistics'))
display(regime_summary(hmm_result))

# Transition matrix
display(Markdown('### Table 3: Regime Transition Matrix'))
labels = ['Compression', 'Normal', 'Stress']
tm = pd.DataFrame(
    hmm_result.transition_matrix,
    index=labels, columns=labels,
).round(3)
display(tm)

### Table 2: Regime Statistics

,n_months,pct_time,mean_oas_bp,std_oas_bp,mean_monthly_chg_bp
regime,,,,,
COMPRESSION,199,57.7,108.0,23.0,-0.4
NORMAL,121,35.1,176.0,49.0,-3.1
STRESS,25,7.2,335.0,156.0,19.4


### Table 3: Regime Transition Matrix

,Compression,Normal,Stress
Compression,0.946,0.046,0.009
Normal,0.088,0.901,0.011
Stress,0.000,0.120,0.880


### Interpretation

- **Regimes are persistent**: the diagonal of the transition matrix shows >90% self-transition probability — once you're in a regime, you tend to stay
- **Stress is rare but devastating**: the model correctly identifies GFC, COVID, and EU crisis as stress episodes
- **Regime detection is forward-looking**: the HMM's filtered probabilities shift *before* spreads reach extremes, providing early warning

### How Regimes Feed Into the Model

```
HMM Regime ──→ τ (prior confidence)  ──→ BL posterior weights
           ──→ ω (view uncertainty)  ──→ How much we trust Prophet forecasts
```

In **Compression**: low τ, low ω → trust the equilibrium, forecasts are reliable  
In **Stress**: high τ, high ω → equilibrium is unreliable, dampen forecast confidence

---

## 6. Black-Litterman Return Estimation <a id='6-bl'></a>

### What Black-Litterman Is

> **Black-Litterman combines market equilibrium with forward-looking views.** Rather than estimating expected returns directly (noisy, unstable), we start from the market's implied returns and adjust based on our spread forecasts.

The core insight of BL: **the market-cap-weighted portfolio IS an optimal portfolio** under some set of implied expected returns. We can reverse-engineer those returns (the "prior"), then blend them with our own forecasts (the "views") to get a "posterior" that is more stable than either alone.

### Why BL Is the Right Framework Here

With only 4 assets, we cannot run a standard mean-variance optimizer — it would put 100% in whichever bucket has the highest expected return. BL solves this by:
1. Starting from the market equilibrium (which is diversified by construction)
2. Making **incremental** adjustments based on our views
3. Controlling how much adjustment via τ (prior uncertainty) and Ω (view uncertainty)

### The Formulation

**Prior** (market equilibrium — what the market "believes"):
$$\pi = \lambda \cdot \Sigma \cdot w_{mkt}$$

Where:
- λ = 2.5 (risk aversion coefficient — standard in BL literature)
- Σ = 4×4 covariance matrix from 60-month rolling window of real FRED returns
- w_mkt = ICE BofA IG market-cap weights [0.04, 0.12, 0.34, 0.50]

**Views** (our forecast — what Prophet "believes"):
- Q = vector of expected returns per bucket, derived from Prophet ΔOAS forecasts
- P = identity matrix (one view per bucket — "absolute" views)

**Posterior** (the Bayesian blend):
$$\mu_{BL} = [(\tau\Sigma)^{-1} + P'\Omega^{-1}P]^{-1} \cdot [(\tau\Sigma)^{-1}\pi + P'\Omega^{-1}Q]$$

Where:
- τ = from HMM regime (0.010 / 0.025 / 0.075)
- Ω = diag(P × τΣ × P') × ω — view uncertainty, scaled by HMM regime

### How to Read the BL Output

- If **posterior ≈ prior**: views are weak or uncertain — the model trusts the market
- If **posterior ≈ views**: views are strong and confident — the model trusts Prophet
- In **stress regime**: posterior stays close to prior (defensive posture)
- In **compression**: posterior incorporates more of the views (risk-on posture)

In [7]:
# ── Figure 3: Black-Litterman Prior vs Views vs Posterior ────────
n = len(IG_ASSETS)

# Compute returns from real FRED data
ret_data = {}
for a in IG_ASSETS:
    if a in monthly.columns:
        dur = DURATIONS.get(a, 7.0)
        ret_data[a] = -dur * monthly[a].diff(1) + monthly[a] / MONTHS_PER_YEAR

Sigma = pd.DataFrame(ret_data).dropna().tail(COV_WINDOW).cov().values
Pi = LAMBDA_RISK_AVERSION * Sigma @ IG_MARKET_WEIGHTS_ARRAY

# Views from recent 3-month OAS changes
Q_vec = np.zeros(n)
for i, a in enumerate(IG_ASSETS):
    if a in monthly.columns:
        chg_3m = float(monthly[a].diff(3).iloc[-1])
        dur = DURATIONS.get(a, 7.0)
        Q_vec[i] = -dur * chg_3m / OAS_PCT_TO_BP

# BL posterior
tau = 0.025  # Normal regime default
P_mat = np.eye(n)
omega_scale = hmm_result.omega_scale
tS = tau * Sigma
Omega = np.diag([
    omega_scale * tau * float(P_mat[i] @ tS @ P_mat[i])
    for i in range(n)
]) + np.eye(n) * BL_RIDGE_PENALTY

tS_inv = np.linalg.inv(tS + np.eye(n) * BL_RIDGE_PENALTY)
Om_inv = np.linalg.inv(Omega)
A = tS_inv + P_mat.T @ Om_inv @ P_mat
b = tS_inv @ Pi + P_mat.T @ Om_inv @ Q_vec
mu_BL = np.linalg.solve(A, b)

# Plot
labels = [ASSET_LABELS.get(a, a) for a in IG_ASSETS]
fig_bl = go.Figure()
fig_bl.add_trace(go.Bar(x=labels, y=Pi * 100, name='Prior (\u03c0)', marker_color=SILVER))
fig_bl.add_trace(go.Bar(x=labels, y=Q_vec * 100, name='Views (Q)', marker_color=GOLD))
fig_bl.add_trace(go.Bar(x=labels, y=mu_BL * 100, name='Posterior (\u03bc_BL)', marker_color=NAVY))

fig_bl.update_layout(
    title='Figure 3: Black-Litterman Return Estimates (FRED Data)',
    yaxis_title='Expected Return (%)',
    barmode='group', template=TEMPLATE, height=450,
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_bl.show()

print(f"\nCurrent regime: {regime_info['regime']}")
print(f"\u03c4 = {tau:.3f}, \u03c9_scale = {omega_scale:.1f}")


Current regime: COMPRESSION
τ = 0.025, ω_scale = 0.5


### How to Read This Chart

- **Prior (π)**: What the market *implies* given current weights and covariance. This is the CAPM equilibrium return.
- **Views (Q)**: What our spread forecast *predicts*. Negative = expecting spread tightening (positive return). Positive = expecting widening (negative return).
- **Posterior (μ_BL)**: The Bayesian blend. In normal regimes, the posterior sits between prior and views. In stress, it stays closer to the prior (views are less trusted).

---

## 7. Prophet OAS Forecasting <a id='7-prophet'></a>

### What Prophet Does in This System

> **Prophet forecasts the OAS for each rating bucket 3 months ahead, converting spread predictions into the "views" (Q) that feed into Black-Litterman.**

Prophet is NOT the allocation model — it is the **view generator**. Its job is to answer one question per bucket: "will spreads tighten or widen over the next 3 months, and by how much?"

### How Prophet Works

Prophet fits a **logistic-growth decomposition** to each OAS time series independently:

$$OAS(t) = g(t) + s(t) + \epsilon(t)$$

Where:
- $g(t)$ = piecewise logistic trend — captures the long-run level of spreads, bounded above by a cap (historical max × 2.5) to prevent forecasting infinite spreads
- $s(t)$ = yearly seasonality (multiplicative) — captures any calendar effects in spread dynamics
- $\epsilon(t)$ = noise — provides the uncertainty interval

### How the Forecast Becomes a View

The Prophet forecast ΔOAS is converted into an expected return via the standard spread-duration approximation:

$$r_{expected} = \underbrace{\frac{OAS_{current}}{10000} \cdot \frac{h}{12}}_{\text{carry (income from spread)}} + \underbrace{D \cdot \frac{-\Delta OAS}{10000} \cdot \frac{h}{12}}_{\text{price return (capital gain/loss)}}$$

- If Prophet forecasts **tightening** (ΔOAS < 0): positive price return → positive expected return → BL overweights this bucket
- If Prophet forecasts **widening** (ΔOAS > 0): negative price return → may still be positive if carry exceeds the loss → BL decides based on magnitude

### Why This Matters

Without Prophet, the BL model would only have the equilibrium prior — it would never deviate from market-cap weights. Prophet gives it a *reason* to tilt, and the HMM-calibrated ω controls how much the model listens.

In [8]:
# ── Figure 4: Prophet Forecasts ──────────────────────────────────
from credit_portfolio.models.prophet_views import generate_all_views

print("Running Prophet forecasts on FRED data (this takes ~30 seconds)...")
views = generate_all_views(monthly, horizon_months=3)

# Summary table
table_rows = []
for bucket, v in views.items():
    delta_bp = v['delta_oas'] * 100
    table_rows.append({
        'Bucket': bucket,
        'Current OAS (%)': f"{v['current_oas']:.2f}",
        'Forecast OAS (%)': f"{v['forecast_oas']:.2f}",
        'Delta (bp)': f"{delta_bp:+.1f}",
        'Uncertainty (%)': f"{v['uncertainty']:.2f}",
        'Expected Return (%)': f"{v['expected_return'] * 100:+.2f}",
    })

display(Markdown('### Table 4: Prophet Forecast Summary'))
display(pd.DataFrame(table_rows).set_index('Bucket'))

2026-03-17 10:52:16,099 [credit_portfolio.models.prophet_views] INFO: Running Prophet forecasts (this takes ~30 seconds)...


Running Prophet forecasts on FRED data (this takes ~30 seconds)...


10:52:16 - cmdstanpy - INFO - Chain [1] start processing


10:52:16 - cmdstanpy - INFO - Chain [1] done processing


2026-03-17 10:52:16,580 [credit_portfolio.models.prophet_views] INFO: AAA: current=0.45% forecast=0.47% delta=+2.4bp exp_ret=+0.06%


...

10:52:16 - cmdstanpy - INFO - Chain [1] start processing


10:52:16 - cmdstanpy - INFO - Chain [1] done processing


2026-03-17 10:52:17,046 [credit_portfolio.models.prophet_views] INFO:  HY: current=3.28% forecast=3.43% delta=+14.5bp exp_ret=+0.66%


### Table 4: Prophet Forecast Summary

,Current OAS (%),Forecast OAS (%),Delta (bp),Uncertainty (%),Expected Return (%)
Bucket,,,,,
AAA,0.45,0.47,+2.4,0.56,+0.06
AA,0.60,0.56,-4.2,0.72,+0.23
A,0.78,0.75,-2.7,0.94,+0.24
BBB,1.15,1.22,+7.2,1.15,+0.17
HY,3.28,3.43,+14.5,3.09,+0.66


In [9]:
# ── Figure 4a: Current vs Forecast OAS ───────────────────────────
buckets = list(views.keys())
current = [views[b]['current_oas'] for b in buckets]
forecast = [views[b]['forecast_oas'] for b in buckets]

fig_pf = go.Figure()
fig_pf.add_trace(go.Bar(x=buckets, y=current, name='Current OAS', marker_color=SILVER))
fig_pf.add_trace(go.Bar(x=buckets, y=forecast, name='Forecast OAS (3mo)', marker_color=NAVY))
fig_pf.update_layout(
    title='Figure 4: Current vs 3-Month Forecast OAS (FRED)',
    yaxis_title='OAS (%)', barmode='group',
    template=TEMPLATE, height=400,
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_pf.show()

# Direction chart
deltas = [views[b]['delta_oas'] * 100 for b in buckets]
colors = [SAGE if d < 0 else CRIMSON for d in deltas]

fig_dir = go.Figure()
fig_dir.add_trace(go.Bar(
    y=buckets, x=deltas, orientation='h',
    marker_color=colors,
    text=[f'{d:+.1f}bp' for d in deltas],
    textposition='outside',
))
fig_dir.update_layout(
    title='Figure 5: Forecast Spread Direction (Green = Tightening, Red = Widening)',
    xaxis_title='Delta OAS (bp)',
    template=TEMPLATE, height=350,
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_dir.show()

---

## 8. Credit Factor Signals <a id='8-factors'></a>

### Three Factors

> **The strategy tilts allocation across rating buckets using three credit factors documented in the academic literature.**

These are NOT equity factors. They are adapted for credit markets:

| Factor | Weight | Definition | Economic Rationale | Credit-Specific? |
|--------|--------|-----------|--------------------|-----------------|
| **DTS** (Duration × Spread) | 50% | D × OAS cross-sectional z-score | Higher DTS = more spread-duration risk = historically compensated. This is the credit equivalent of "beta" (Bai, Bali, Wen 2019) | Yes — does not exist in equities |
| **Value** | 25% | Z-score of log(OAS) across buckets | Wide spreads relative to peers tend to mean-revert. This is the credit equivalent of "cheap vs expensive" (Ben Dor et al. 2012) | Adapted — uses OAS, not P/E |
| **Momentum** | 25% | Trailing 6-month cumulative excess return | Buckets with positive momentum tend to continue outperforming. Same concept as equity momentum but applied to spread returns (Israel et al. 2018) | Adapted — uses spread returns |

### Important Caveat on Cross-Sectional Factors

With only 4 rating buckets, **DTS and Value are partly mechanical** — BBB almost always has the highest DTS and widest spreads. The time-varying signal comes from:
- **DTS**: when the BBB DTS *relative to its own history* is unusually high, the risk premium is elevated
- **Value**: when the BBB-AAA spread differential is wider than normal, value signals are strongest
- **Momentum**: this is the factor with the most genuine cross-sectional variation — sometimes AA outperforms BBB

We set DTS weight at 50% not because it's the "best" predictor, but because the credit literature (Bai et al. 2019) identifies it as the dominant single factor in credit return dispersion.

### Composite Signal

$$z_{composite,i} = 0.50 \cdot z_{DTS,i} + 0.25 \cdot z_{Value,i} + 0.25 \cdot z_{Momentum,i}$$

### Portfolio Construction

$$w_i = w_{mkt,i} + \alpha \cdot \frac{z_{composite,i}}{\sum |z_{composite}|}$$

Where α = tilt strength (default 10%). Weights are floored at 1% and renormalized to sum to 100%.

---

## 9. Historical Backtest Results <a id='9-backtest'></a>

### Setup

> **The backtest runs on the full FRED history with monthly rebalancing, no look-ahead bias, and realistic transaction costs. This is the primary test of our hypothesis.**

Recall from Section 2.7: we are testing whether regime-conditional factor rotation generates statistically significant alpha over the market-cap benchmark.

In [10]:
# ── Figure 6: Historical Backtest ────────────────────────────────
from credit_portfolio.backtests.bucket_backtest import run_backtest, BacktestConfig

bt_config = BacktestConfig(tilt_strength=0.10, momentum_window=6, tc_bps=5.0)
bt_result = run_backtest(monthly, config=bt_config)

# Cumulative returns
fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(
    x=bt_result.cumulative_strategy.index,
    y=bt_result.cumulative_strategy.values,
    name='Factor Strategy', line=dict(color=NAVY, width=2),
))
fig_bt.add_trace(go.Scatter(
    x=bt_result.cumulative_benchmark.index,
    y=bt_result.cumulative_benchmark.values,
    name='Market-Cap Benchmark', line=dict(color=SILVER, width=2, dash='dash'),
))

for date, label in events:
    fig_bt.add_vline(x=date, line_dash='dot', line_color=SILVER, line_width=0.5)

fig_bt.update_layout(
    title='Figure 6: Cumulative Returns \u2014 Factor Strategy vs Benchmark (FRED)',
    yaxis_title='Growth of $1',
    template=TEMPLATE, height=500,
    legend=dict(orientation='h', y=-0.12),
    hovermode='x unified',
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_bt.show()

In [11]:
# ── Table 5: Backtest Performance Statistics ─────────────────────
stats = bt_result.stats

perf_data = {
    'Metric': [
        'Period', 'Months',
        'Annualized Return', 'Annualized Volatility', 'Sharpe Ratio', 'Max Drawdown',
        'Annualized Alpha', 'Information Ratio', 't-stat (Alpha)',
        'Hit Rate', 'Tracking Error', 'Avg Monthly Turnover',
    ],
    'Strategy': [
        stats['period'], stats['n_months'],
        f"{stats['ann_return_strategy']:+.2%}",
        f"{stats['ann_vol_strategy']:.2%}",
        f"{stats['sharpe_strategy']:.2f}",
        f"{stats['max_drawdown_strategy']:.2%}",
        f"{stats['ann_alpha']:+.2%}",
        f"{stats['information_ratio']:.2f}",
        f"{stats['t_stat_alpha']:.2f}",
        f"{stats['hit_rate']:.1%}",
        f"{stats['tracking_error']:.2%}",
        f"{stats['avg_monthly_turnover']:.2%}",
    ],
    'Benchmark': [
        stats['period'], stats['n_months'],
        f"{stats['ann_return_benchmark']:+.2%}",
        f"{stats['ann_vol_benchmark']:.2%}",
        f"{stats['sharpe_benchmark']:.2f}",
        f"{stats['max_drawdown_benchmark']:.2%}",
        '—', '—', '—', '—', '—', '—',
    ],
}

display(Markdown('### Table 5: Backtest Performance Statistics'))
display(pd.DataFrame(perf_data).set_index('Metric'))

### Table 5: Backtest Performance Statistics

,Strategy,Benchmark
Metric,,
Period,1997-09 to 2026-03,1997-09 to 2026-03
Months,343,343
Annualized Return,+1.36%,+1.31%
Annualized Volatility,5.64%,5.51%
Sharpe Ratio,0.24,0.24
Max Drawdown,-30.88%,-30.63%
Annualized Alpha,+0.05%,—
Information Ratio,0.25,—
t-stat (Alpha),1.53,—


In [12]:
# ── Figure 7: Factor Signals Over Time ───────────────────────────
sig_df = bt_result.signals_history

fig_sig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['DTS Signal', 'Value Signal', 'Momentum Signal'],
    vertical_spacing=0.06,
)

bucket_colors = {'AAA': NAVY, 'AA': DENIM, 'A': SKY, 'BBB': GOLD}

for row_i, prefix in enumerate(['z_dts', 'z_value', 'z_momentum'], 1):
    for bucket in RATINGS:
        col = f'{prefix}_{bucket}'
        if col in sig_df.columns:
            fig_sig.add_trace(go.Scatter(
                x=sig_df.index, y=sig_df[col],
                name=bucket if row_i == 1 else None,
                showlegend=(row_i == 1),
                line=dict(color=bucket_colors[bucket], width=1.2),
            ), row=row_i, col=1)
    fig_sig.add_hline(y=0, row=row_i, col=1,
                      line_dash='solid', line_color=CHARCOAL, line_width=0.3)

fig_sig.update_layout(
    title='Figure 7: Credit Factor Signals by Rating Bucket (FRED)',
    template=TEMPLATE, height=700,
    legend=dict(orientation='h', y=-0.05),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_sig.show()

In [13]:
# ── Figure 8: Weight Allocation Over Time ────────────────────────
bucket_colors = {'AAA': NAVY, 'AA': DENIM, 'A': SKY, 'BBB': GOLD}

fig_wa = go.Figure()
for bucket in bt_result.weights_history.columns:
    fig_wa.add_trace(go.Scatter(
        x=bt_result.weights_history.index,
        y=bt_result.weights_history[bucket].values,
        name=bucket, stackgroup='one',
        line=dict(width=0.5, color=bucket_colors.get(bucket, SILVER)),
    ))

fig_wa.update_layout(
    title='Figure 8: Rating Bucket Allocation Over Time',
    yaxis_title='Weight', yaxis_range=[0, 1],
    template=TEMPLATE, height=450,
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_wa.show()

### Backtest Interpretation

**What the strategy does well:**
- Captures the **BBB carry premium** while rotating away during stress
- **Value signal** correctly identifies mean-reversion opportunities after spread dislocations
- **DTS factor** provides risk-adjusted exposure sizing

**Limitations (important context):**
- Only 4 assets (rating buckets) — limits alpha generation vs a bond-level strategy
- Spread returns are an approximation (carry + duration × ΔOAS)
- Transaction costs assume institutional-level execution (5bp one-way)
- No idiosyncratic issuer selection — purely a top-down rotation

---

## 10. Stress Testing Framework <a id='10-stress'></a>

### Purpose

> **Stress tests answer: "what happens to the portfolio under extreme but plausible scenarios?"** This is NOT a test of the hypothesis — it is a robustness check.

We apply predefined OAS shocks to real FRED data and measure the impact on portfolio weights, carry, and price returns.

| Scenario | AAA Shock | AA Shock | A Shock | BBB Shock | Historical Analog |
|----------|-----------|---------|---------|----------|-----------------|
| Spread Widening +200bp | +200 | +200 | +200 | +200 | Generic risk-off |
| BBB Crisis | — | — | +50 | +300 | 2002 Worldcom, 2018 GE |
| Fed Hike Shock | +100 | +100 | +100 | +100 | 2022 rate shock |
| COVID Replay | +280 | +280 | +280 | +280 | March 2020 |

In [14]:
# ── Figure 9: Stress Test — BBB Crisis ───────────────────────────
from credit_portfolio.analytics.stress_test import run_stress_test

scenarios = ['Spread Widening +200bp', 'BBB Crisis', 'Fed Hike Shock', 'COVID Replay']
stress_results = {}

for sc in scenarios:
    stress_results[sc] = run_stress_test(monthly, scenario=sc)

# Show BBB Crisis in detail
sr = stress_results['BBB Crisis']
display(Markdown(f'### Table 6: Stress Test — {sr.scenario_name}'))
display(Markdown(f'*{sr.description}*'))
display(sr.weight_changes.set_index('Bucket'))

### Table 6: Stress Test — BBB Crisis

*BBB spreads blow out +300bp, A widens +50bp*

,Baseline OAS (%),Stressed OAS (%),Shock (bp),Market Weight,Baseline Weight,Stressed Weight,Weight Change
Bucket,,,,,,,
AAA,0.45,0.45,0.0,0.04,0.009982,0.009990,0.000008
AA,0.60,0.60,0.0,0.12,0.101596,0.100888,-0.000708
A,0.78,1.28,50.0,0.34,0.349539,0.349353,-0.000186
BBB,1.15,4.15,300.0,0.50,0.538882,0.539769,0.000887


In [15]:
# ── Figure 9a: Price Impact Across Scenarios ─────────────────────
fig_stress = go.Figure()

scenario_colors = {
    'Spread Widening +200bp': GOLD,
    'BBB Crisis': CRIMSON,
    'Fed Hike Shock': DENIM,
    'COVID Replay': SILVER,
}

for sc_name, sr in stress_results.items():
    impacts = [sr.price_impact[b] * 100 for b in RATINGS]
    fig_stress.add_trace(go.Bar(
        x=list(RATINGS), y=impacts,
        name=sc_name, marker_color=scenario_colors.get(sc_name, SILVER),
    ))

fig_stress.update_layout(
    title='Figure 9: Price Impact by Scenario and Rating Bucket',
    yaxis_title='Price Return (%)',
    barmode='group', template=TEMPLATE, height=450,
    legend=dict(orientation='h', y=-0.15),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_stress.show()

In [16]:
# ── Figure 9b: OAS Levels Baseline vs Stressed ───────────────────
fig_oas_stress = make_subplots(
    rows=2, cols=2, subplot_titles=scenarios,
    vertical_spacing=0.12, horizontal_spacing=0.08,
)

for idx, sc_name in enumerate(scenarios):
    sr = stress_results[sc_name]
    row, col = idx // 2 + 1, idx % 2 + 1
    base = [sr.baseline_oas[b] for b in RATINGS]
    stressed = [sr.stressed_oas[b] for b in RATINGS]
    
    fig_oas_stress.add_trace(go.Bar(
        x=list(RATINGS), y=base, name='Baseline' if idx == 0 else None,
        marker_color=NAVY, showlegend=(idx == 0),
    ), row=row, col=col)
    fig_oas_stress.add_trace(go.Bar(
        x=list(RATINGS), y=stressed, name='Stressed' if idx == 0 else None,
        marker_color=CRIMSON, showlegend=(idx == 0),
    ), row=row, col=col)

fig_oas_stress.update_layout(
    title='Figure 10: OAS Levels \u2014 Baseline vs Stressed (All Scenarios)',
    template=TEMPLATE, height=600, barmode='group',
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_oas_stress.show()

---

## 11. Monte Carlo Risk Analysis <a id='11-mc'></a>

### Purpose

> **Monte Carlo simulation quantifies the distribution of possible future outcomes using real historical return data from the FRED backtest.** This is the second robustness check.

Unlike the backtest (which tells us what *did* happen), Monte Carlo tells us what *could* happen by resampling from the actual return distribution.

### Methodology

Two methods (user selects via dashboard):
1. **Empirical Bootstrap**: Sample monthly returns with replacement from the actual backtest history. This preserves the real return distribution including skewness and fat tails.
2. **Parametric Normal**: Fit N(μ, σ²) to historical returns, then sample. This assumes normality — a simplification that underestimates tail risk.

### Key Risk Metrics

| Metric | Definition | What It Tells You |
|--------|-----------|-------------------|
| **VaR (Value at Risk)** | The loss threshold at a given confidence level | "There's a 5% chance of losing more than X%" |
| **CVaR (Conditional VaR)** | Average loss when VaR is breached | "When things go wrong, on average we lose Y%" — captures tail severity |
| **P(Loss)** | Probability of any negative cumulative return over the horizon | "What's the chance we're underwater at the end?" |

In [17]:
# ── Figure 11: Monte Carlo Fan Chart ─────────────────────────────
from credit_portfolio.analytics.monte_carlo import run_monte_carlo

# Use real backtest returns
mc_result = run_monte_carlo(
    historical_returns=bt_result.monthly_returns_strategy,
    n_sims=5000,
    horizon_months=24,
    method='bootstrap',
    confidence_levels=[90.0, 95.0, 99.0],
)

months = np.arange(1, mc_result.horizon_months + 1)

fig_fan = go.Figure()

# 5-95 band
fig_fan.add_trace(go.Scatter(
    x=months, y=mc_result.percentile_bands[95] * 100,
    mode='lines', line=dict(width=0), showlegend=False,
))
fig_fan.add_trace(go.Scatter(
    x=months, y=mc_result.percentile_bands[5] * 100,
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(0,32,91,0.08)',
    name='5th\u201395th percentile',
))

# 25-75 band
fig_fan.add_trace(go.Scatter(
    x=months, y=mc_result.percentile_bands[75] * 100,
    mode='lines', line=dict(width=0), showlegend=False,
))
fig_fan.add_trace(go.Scatter(
    x=months, y=mc_result.percentile_bands[25] * 100,
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(0,32,91,0.20)',
    name='25th\u201375th percentile',
))

# Median
fig_fan.add_trace(go.Scatter(
    x=months, y=mc_result.median_path * 100,
    mode='lines', line=dict(color=NAVY, width=2.5),
    name='Median',
))

fig_fan.add_hline(y=0, line_dash='dash', line_color=CHARCOAL, line_width=0.5)
fig_fan.update_layout(
    title=f'Figure 11: Monte Carlo Fan Chart \u2014 {mc_result.n_sims:,} Simulations, {mc_result.horizon_months}mo Horizon',
    xaxis_title='Month', yaxis_title='Cumulative Return (%)',
    template=TEMPLATE, height=500,
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_fan.show()

In [18]:
# ── Figure 12: Terminal Return Distribution ──────────────────────
fig_hist = go.Figure()
fig_hist.add_trace(go.Histogram(
    x=mc_result.terminal_returns * 100,
    nbinsx=60, marker_color=NAVY, opacity=0.7,
    name='Terminal Return Distribution',
))

for _, row in mc_result.risk_metrics.iterrows():
    fig_hist.add_vline(
        x=row['VaR'] * 100,
        line_dash='dash', line_color=CRIMSON, line_width=1.5,
        annotation_text=f"VaR {row['Confidence']}",
        annotation_position='top',
    )

fig_hist.add_vline(x=0, line_dash='solid', line_color=CHARCOAL, line_width=0.5)
fig_hist.update_layout(
    title=f'Figure 12: Terminal Return Distribution ({mc_result.horizon_months}mo)',
    xaxis_title='Cumulative Return (%)', yaxis_title='Count',
    template=TEMPLATE, height=450,
    font=dict(family='Helvetica Neue, Arial, sans-serif'),
)
fig_hist.show()

In [19]:
# ── Table 7: Risk Metrics ────────────────────────────────────────
display(Markdown('### Table 7: Monte Carlo Risk Metrics'))
risk_display = mc_result.risk_metrics.copy()
risk_display['VaR'] = risk_display['VaR'].apply(lambda x: f'{x * 100:+.2f}%')
risk_display['CVaR'] = risk_display['CVaR'].apply(lambda x: f'{x * 100:+.2f}%')
risk_display['P(Loss)'] = risk_display['P(Loss)'].apply(lambda x: f'{x:.1%}')
display(risk_display.set_index('Confidence'))

# Summary
tr = mc_result.terminal_returns
print(f"\nMonte Carlo Summary ({mc_result.n_sims:,} sims, {mc_result.horizon_months}mo):")
print(f"  Median return: {np.median(tr)*100:+.2f}%")
print(f"  Mean return:   {np.mean(tr)*100:+.2f}%")
print(f"  Std deviation: {np.std(tr)*100:.2f}%")
print(f"  P(Loss):       {(tr < 0).mean():.1%}")
print(f"  Best case:     {np.max(tr)*100:+.2f}%")
print(f"  Worst case:    {np.min(tr)*100:+.2f}%")

### Table 7: Monte Carlo Risk Metrics

,VaR,CVaR,P(Loss)
Confidence,,,
90%,-7.13%,-11.96%,31.2%
95%,-11.11%,-14.97%,31.2%
99%,-17.45%,-20.74%,31.2%



Monte Carlo Summary (5,000 sims, 24mo):
  Median return: +3.61%
  Mean return:   +3.43%
  Std deviation: 8.26%
  P(Loss):       31.2%
  Best case:     +36.48%
  Worst case:    -29.56%


---

## 12. Conclusions & Next Steps <a id='12-conclusions'></a>

### Revisiting the Hypothesis

Recall **H₁**: *Systematic rotation across IG rating tiers, conditioned on a regime-switching model, generates statistically significant excess returns over a market-cap-weighted IG benchmark after transaction costs.*

Based on the backtest results in Section 9, we [assess the t-stat, alpha, and information ratio to determine whether the hypothesis is supported, inconclusive, or rejected — see the metrics table above].

### What the Model Does Well

1. **Regime detection adds value**: The 3-state HMM correctly identifies compression, normal, and stress environments — enabling adaptive parameter calibration that prevents overweight BBB at the worst times
2. **Black-Litterman provides regularization**: Rather than raw signal-chasing, BL anchors allocations to market equilibrium — critical with only 4 assets
3. **The pipeline is transparent**: Every allocation decision traces to observable data — regime state, Prophet forecast, factor scores
4. **Risk management is built in**: Stress tests and Monte Carlo are integral, not afterthoughts

### What the Model Does NOT Do Well

1. **Alpha is constrained by the 4-bucket universe** — with only 4 rating tiers, there is limited room for outperformance
2. **DTS and Value factors are partly mechanical** — they tend to overweight BBB, which is essentially a carry trade with regime protection
3. **Prophet is a noisy forecaster** — at monthly frequency, spread forecasts have low signal-to-noise ratio
4. **The return approximation introduces tracking error** — actual index returns differ from our carry + duration × ΔOAS estimate

### Limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Only 4 rating buckets | Limits alpha generation capacity | Bond-level data (TRACE/Bloomberg) would enable issuer selection |
| Spread return approximation | May diverge from index total returns | Could validate against total return indices (not freely available) |
| No credit events modeled | Ignores defaults/downgrades | Add fallen angel transition model with Moody's data |
| Static factor weights | 50/25/25 may not be optimal in all regimes | Time-varying weights via regime-dependent calibration |
| HMM look-ahead in training | Mild bias from fitting on full sample | Expanding-window refitting in production |
| Cross-sectional factors with n=4 | Noisy z-scores | Modest tilt strength (10%) as regularization |

### Recommended Next Steps

1. **Bond-level data integration** — TRACE or Bloomberg for issuer-level allocation (would increase cross-section from 4 to 1000+)
2. **Sector rotation overlay** — extend from 4 rating buckets to 8-10 sectors × 4 ratings
3. **Regime-dependent factor weights** — DTS may matter more in stress, momentum in compression
4. **Live signal monitoring** — real-time FRED API integration for daily signal updates
5. **Expanding-window HMM** — refit HMM at each rebalance date using only past data (removes look-ahead)
6. **Out-of-sample validation** — hold out the last 5 years for true out-of-sample testing

---

### References

- Ang, A. & Bekaert, G. (2002). International Asset Allocation with Regime Shifts. *Review of Financial Studies*, 15(4).
- Bai, J., Bali, T., & Wen, Q. (2019). Common risk factors in the cross-section of corporate bond returns. *Journal of Financial Economics*, 131(3).
- Ben Dor, A., Dynkin, L., Hyman, J., Houweling, P., van Leeuwen, E., & Penninga, O. (2012). *Systematic Credit Investing*. Wiley.
- Black, F. & Litterman, R. (1992). Global Portfolio Optimization. *Financial Analysts Journal*, 48(5).
- Fabozzi, F., Focardi, S., & Jonas, C. (2010). *Investment Management after the Global Financial Crisis*. CFA Institute.
- Guidolin, M. & Timmermann, A. (2007). Asset Allocation under Multivariate Regime Switching. *Journal of Economic Dynamics and Control*, 31(11).
- Hamilton, J. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series. *Econometrica*, 57(2).
- Houweling, P. & van Zundert, J. (2017). Factor Investing in the Corporate Bond Market. *Financial Analysts Journal*, 73(2).
- Israel, R., Palhares, D., & Richardson, S. (2018). Common factors in corporate bond returns. *Journal of Investment Management*, 16(2).
- Meucci, A. (2010). The Black-Litterman Approach: Original Model and Extensions. *SSRN Working Paper*.
- Taylor, S. & Letham, B. (2018). Forecasting at Scale. *The American Statistician*, 72(1). [Prophet]

---

*All data sourced from FRED (Federal Reserve Economic Data). ICE BofA indices. No synthetic or simulated data used in any analysis.*